# IEEE 754: 부동소수점의 이진 내부 구조 - IEEE 754 비트 레이아웃과 정밀도 한계

- Tutorial ID: `ull-1`
- Tutorial: IEEE 754: 부동소수점의 이진 내부 구조
- Section ID: `ull-1-1`
- Section: IEEE 754 비트 레이아웃과 정밀도 한계


In [ ]:
# ============================================================
# 코드 읽는 법 — IEEE 754 비트 레이아웃과 정밀도 한계
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 62)
print("IEEE 754 부동소수점: 비트 수준 분석")
print("=" * 62)

In [ ]:
# 1. FP32 비트 분해

In [ ]:
print("\n1. FP32 비트 분해")
print("-" * 50)

import struct

def float_to_bits(f):
    """FP32를 32비트 이진 문자열로 변환"""
    packed = struct.pack('>f', f)
    bits = ''.join(f'{b:08b}' for b in packed)
    return bits

def bits_to_components(bits):
    """32비트를 부호/지수/가수로 분해"""
    sign = int(bits[0])
    exponent_bits = bits[1:9]
    mantissa_bits = bits[9:32]
    exponent = int(exponent_bits, 2)
    # 가수: 1.mantissa (암묵적 1)
    mantissa = 1.0
        for i, b in enumerate(mantissa_bits):
        if b == '1':
            mantissa += 2 ** -(i + 1)
    return sign, exponent, exponent_bits, mantissa_bits, mantissa

test_values = [1.0, -1.0, 0.5, 3.14, 0.1, 100.0, 1e-7, 1e38]

for val in test_values:
    bits = float_to_bits(val)
    sign, exp, exp_bits, man_bits, mantissa = bits_to_components(bits)
    
    sign_str = '+' if sign == 0 else '-'
    actual_exp = exp - 127
    
    print(f"\n  {val:>12g} =")
    print(f"    비트: [{bits[0]}] [{exp_bits}] [{man_bits}]")
    print(f"          S     E(8bit)     M(23bit)")
    print(f"    분해: {sign_str}1 × 2^({exp}-127) × {mantissa:.6f}")
    print(f"         = {sign_str}{mantissa:.6f} × 2^{actual_exp}")

In [ ]:
# 2. 정밀도 한계 시각화

In [ ]:
print("\n\n2. 정밀도 한계")
print("-" * 50)

print("\n가수(23비트)가 표현할 수 있는 정밀도:")
print(f"  2^(-23) = {2**-23:.10f}")
print(f"  ≈ 1.19e-7 (약 7자리 십진수 정밀도)")

# 정밀도 한계 시연
a = np.float32(1.0)
for delta in [1e-6, 1e-7, 1e-8, 1e-9]:
        result = np.float32(a + delta) - a
    lost = "OK" if result > 0 else "LOST!"
    print(f"  1.0 + {delta:.0e} - 1.0 = {result:.2e}  [{lost}]")

In [ ]:
# 3. FP32 vs FP16 vs BF16 비트 비교

In [ ]:
print("\n3. 포맷 비교 (비트 레이아웃)")
print("-" * 50)

formats = [
    ("FP32", 1, 8, 23, 127),
    ("FP16", 1, 5, 10, 15),
    ("BF16", 1, 8, 7, 127),
]

print(f"  {'Format':>6}  {'Sign':>4}  {'Exp':>5}  {'Man':>5}  {'Bias':>5}  {'Range':>12}  {'Precision':>10}")
print("  " + "-" * 60)
for name, s, e, m, bias in formats:
    max_exp = (2**e - 1) - bias
    min_exp = 1 - bias
    max_val = (2 - 2**-m) * 2**max_exp
    eps = 2**-m
    print(f"  {name:>6}  {s:>4}  {e:>5}  {m:>5}  {bias:>5}  2^{max_exp:<5}  {eps:.2e}")

print("""
  시각적 비교:
  FP32: [S][EEEEEEEE][MMMMMMMMMMMMMMMMMMMMMMM]   32비트
  FP16: [S][EEEEE][MMMMMMMMMM]                   16비트
  BF16: [S][EEEEEEEE][MMMMMMM]                   16비트
  
  BF16이 FP32와 같은 지수 범위를 가지는 이유:
  → 지수 비트가 동일! (8비트)
  → 가수만 줄임 (23→7비트)
  → 범위 유지, 정밀도만 감소
""")

In [ ]:
# 4. 비정규수와 특수값

In [ ]:
print("4. 특수값")
print("-" * 50)

special = [
    (0.0, "양의 영"),
    (-0.0, "음의 영"),
    (float('inf'), "양의 무한대"),
    (float('-inf'), "음의 무한대"),
    (float('nan'), "NaN"),
]

for val, name in special:
    try:
        bits = float_to_bits(val)
        print(f"  {name:>12}: [{bits[0]}][{bits[1:9]}][{bits[9:]}]")
    except:
        print(f"  {name:>12}: (표현 불가)")

print("""
  규칙:
  - 지수=0, 가수=0 → 0 (부호에 따라 +0, -0)
  - 지수=255, 가수=0 → ±Infinity
  - 지수=255, 가수≠0 → NaN
  - 지수=0, 가수≠0 → 비정규수 (subnormal)
""")

In [ ]:
# 5. 트랜스포머에서의 수치 문제

In [ ]:
print("5. 트랜스포머 수치 문제 사례")
print("-" * 50)

# 소프트맥스 오버플로우
x = np.array([1000.0, 1001.0, 1002.0], dtype=np.float32)
print(f"  입력: {x}")

with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(x)
    print(f"  exp(x): {naive}  ← 오버플로우!")

x_shifted = x - np.max(x)
stable = np.exp(x_shifted)
result = stable / stable.sum()
print(f"  exp(x - max): {np.round(stable, 4)}")
print(f"  softmax:      {np.round(result, 4)}  ← 정상")

# 그래디언트 언더플로우
print(f"\n  FP16 최소 양수: {np.finfo(np.float16).tiny:.2e}")
print(f"  FP32 최소 양수: {np.finfo(np.float32).tiny:.2e}")
print(f"  전형적 그래디언트: 1e-4 ~ 1e-8")
print(f"  → FP16에서 1e-8 이하 그래디언트 = 0! (정보 손실)")